In [11]:
import warnings
import re
warnings.filterwarnings(
    "ignore",
    message=re.escape("The 'repr' attribute with value False was provided to the `Field()` function"),
)

warnings.filterwarnings(
    "ignore",
    message=re.escape("The 'frozen' attribute with value True was provided to the `Field()` function"),
)
warnings.filterwarnings("ignore", message=".*Mapping deprecated model name.*")
warnings.filterwarnings("ignore", message=".*Palette images with Transparency expressed in bytes.*")
warnings.filterwarnings("ignore", category=FutureWarning, message=".*autocast.*")
warnings.filterwarnings("ignore", category=FutureWarning, message=".*GradScaler.*")

In [1]:
%%writefile data_split.py
import shutil
from pathlib import Path
import random
from tqdm import tqdm

# ============================
# HARD-CODED CONFIG
# ============================

SRC_DIRS = [
    "/kaggle/input/ai-generated-images-vs-real-images/AiArtData",
    "/kaggle/input/ai-generated-images-vs-real-images/RealArt",
]

DST_ROOT = "/kaggle/working/data"
VAL_SIZE = 0.1
TEST_SIZE = 0.1
SEED = 42

CLASS_MAP = {
    "aiartdata": "ai",
    "realart": "real"
}

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff"}

# ============================
# SCRIPT START
# ============================

def gather_images(src_dirs):
    images = []
    for d in src_dirs:
        p = Path(d)
        if not p.exists():
            print(f"WARNING: {d} does not exist, skipping")
            continue
        for f in p.rglob("*"):
            if f.is_file() and f.suffix.lower() in IMG_EXTS:
                images.append(f)
    return images


def main():
    random.seed(SEED)
    dst_root = Path(DST_ROOT)
    dst_root.mkdir(parents=True, exist_ok=True)

    labeled = {"ai": [], "real": []}

    # Assign files to labels based on folder name
    for src in SRC_DIRS:
        src_path = Path(src)
        base = src_path.name.lower()
        label = CLASS_MAP.get(base, None)

        if label is None:
            print(f"ERROR: Folder {base} not mapped in CLASS_MAP")
            return

        files = gather_images([src_path])
        for f in files:
            labeled[label].append(f)

    # Print dataset stats
    print("Dataset summary:")
    for k, v in labeled.items():
        print(f"{k}: {len(v)} images")

    # Create folders
    for split in ["train", "val", "test"]:
        for lab in labeled.keys():
            (dst_root / split / lab).mkdir(parents=True, exist_ok=True)

    # Split & copy
    def split_copy(files, label):
        random.shuffle(files)
        n = len(files)
        n_test = int(n * TEST_SIZE)
        n_val = int(n * VAL_SIZE)

        test_files = files[:n_test]
        val_files = files[n_test:n_test + n_val]
        train_files = files[n_test + n_val:]

        for f in tqdm(train_files, desc=f"Copying {label} train"):
            shutil.copy(f, dst_root / "train" / label / f.name)

        for f in tqdm(val_files, desc=f"Copying {label} val"):
            shutil.copy(f, dst_root / "val" / label / f.name)

        for f in tqdm(test_files, desc=f"Copying {label} test"):
            shutil.copy(f, dst_root / "test" / label / f.name)

    for label, files in labeled.items():
        split_copy(files, label)

    print("\nFinal counts:")
    for split in ["train", "val", "test"]:
        for lab in labeled.keys():
            count = len(list((dst_root / split / lab).glob("*")))
            print(f"{split}/{lab}: {count}")


if __name__ == "__main__":
    main()


Writing data_split.py


In [2]:
!python data_split.py 

Dataset summary:
ai: 539 images
real: 434 images
Copying real test: 100%|███████████████████████| 43/43 [00:00<00:00, 127.91it/s]

Final counts:
train/ai: 433
train/real: 348
val/ai: 53
val/real: 43
test/ai: 53
test/real: 43


In [12]:
%%writefile train_kaggle.py
# train_kaggle.py
import os
import random
from pathlib import Path
from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
from sklearn.metrics import roc_auc_score, accuracy_score
from torch.cuda.amp import autocast, GradScaler

# ----------------------
# Config (edit if needed)
# ----------------------
DATA_DIR = "/kaggle/working/data"        
OUT_CKPT = "/kaggle/working/best_model.pth"
MODEL_NAME = "tf_efficientnet_b3_ns"    
IMG_SIZE = 300
BATCH_SIZE = 24                         
HEAD_EPOCHS = 3                         
FINE_TUNE_EPOCHS = 10                   # then unfreeze and fine-tune
LR = 3e-4
WEIGHT_DECAY = 1e-4
SEED = 42
NUM_WORKERS = 2
DROPOUT = 0.4
LABEL_SMOOTHING = 0.0                   

# ----------------------
# Utilities
# ----------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def get_transforms(img_size=IMG_SIZE):
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.7,1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomApply([transforms.ColorJitter(0.2,0.2,0.2,0.05)], p=0.7),
        transforms.RandomGrayscale(p=0.02),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ])
    valid_tf = transforms.Compose([
        transforms.Resize(int(img_size*1.05)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ])
    return train_tf, valid_tf

def create_model(model_name=MODEL_NAME, pretrained=True, dropout=DROPOUT):
    # create backbone with no head, add binary head
    backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0, global_pool='avg')
    in_f = backbone.num_features
    head = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(in_f, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout/2),
        nn.Linear(512, 1)
    )
    model = nn.Sequential(backbone, head)
    return model

# ----------------------
# Training/Eval loops
# ----------------------
def train_one_epoch(model, loader, opt, criterion, device, scaler):
    model.train()
    losses = []
    all_preds = []
    all_labels = []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.float().unsqueeze(1).to(device)
        opt.zero_grad()
        with autocast():
            logits = model(imgs)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        losses.append(loss.item())
        preds = torch.sigmoid(logits).detach().cpu().numpy()
        all_preds.extend(preds.ravel().tolist())
        all_labels.extend(labels.detach().cpu().numpy().ravel().tolist())
    auc = roc_auc_score(all_labels, all_preds) if len(set(all_labels))>1 else 0.0
    return np.mean(losses), auc

def validate(model, loader, criterion, device):
    model.eval()
    losses = []
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.float().unsqueeze(1).to(device)
            logits = model(imgs)
            loss = criterion(logits, labels)
            losses.append(loss.item())
            all_preds.extend(torch.sigmoid(logits).cpu().numpy().ravel().tolist())
            all_labels.extend(labels.cpu().numpy().ravel().tolist())
    auc = roc_auc_score(all_labels, all_preds) if len(set(all_labels))>1 else 0.0
    preds_bin = [1 if p>0.5 else 0 for p in all_preds]
    acc = accuracy_score(all_labels, preds_bin)
    return np.mean(losses), auc, acc

# ----------------------
# Main
# ----------------------
def main():
    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    train_tf, valid_tf = get_transforms(IMG_SIZE)
    train_ds = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=train_tf)
    val_ds = datasets.ImageFolder(os.path.join(DATA_DIR, "val"), transform=valid_tf)
    test_ds = datasets.ImageFolder(os.path.join(DATA_DIR, "test"), transform=valid_tf)

    print("Classes:", train_ds.classes)
    print("Train size:", len(train_ds), "Val size:", len(val_ds), "Test size:", len(test_ds))

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    model = create_model(MODEL_NAME, pretrained=True, dropout=DROPOUT)
    model = model.to(device)

    # loss with optional label smoothing for binary (simple approach)
    bce = nn.BCEWithLogitsLoss()

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=HEAD_EPOCHS + FINE_TUNE_EPOCHS)
    scaler = GradScaler()

    best_auc = 0.0

    # ---------- Phase 1: train head only ----------
    print("Phase 1: training head only for", HEAD_EPOCHS, "epochs")
    # freeze backbone (first module)
    backbone = model[0]
    for param in backbone.parameters():
        param.requires_grad = False

    for epoch in range(HEAD_EPOCHS):
        print(f"HEAD epoch {epoch+1}/{HEAD_EPOCHS}")
        train_loss, train_auc = train_one_epoch(model, train_loader, opt, bce, device, scaler)
        val_loss, val_auc, val_acc = validate(model, val_loader, bce, device)
        scheduler.step()
        print(f"Train loss {train_loss:.4f} AUC {train_auc:.4f} | Val loss {val_loss:.4f} AUC {val_auc:.4f} Acc {val_acc:.4f}")
        if val_auc > best_auc:
            best_auc = val_auc
            torch.save({'model_state': model.state_dict(), 'model_name': MODEL_NAME}, OUT_CKPT)
            print("Saved best checkpoint (head phase) to", OUT_CKPT)

    # ---------- Phase 2: unfreeze & fine-tune ----------
    print("Phase 2: unfreeze backbone and fine-tune for", FINE_TUNE_EPOCHS, "epochs")
    for param in backbone.parameters():
        param.requires_grad = True

    # optionally lower LR for finetuning
    opt = torch.optim.AdamW(model.parameters(), lr=LR/3, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=FINE_TUNE_EPOCHS)
    scaler = GradScaler()

    for epoch in range(FINE_TUNE_EPOCHS):
        print(f"FINETUNE epoch {epoch+1}/{FINE_TUNE_EPOCHS}")
        train_loss, train_auc = train_one_epoch(model, train_loader, opt, bce, device, scaler)
        val_loss, val_auc, val_acc = validate(model, val_loader, bce, device)
        scheduler.step()
        print(f"Train loss {train_loss:.4f} AUC {train_auc:.4f} | Val loss {val_loss:.4f} AUC {val_auc:.4f} Acc {val_acc:.4f}")
        if val_auc > best_auc:
            best_auc = val_auc
            torch.save({'model_state': model.state_dict(), 'model_name': MODEL_NAME}, OUT_CKPT)
            print("Saved best checkpoint to", OUT_CKPT)

    # Final test evaluation (load best)
    print("Loading best model from", OUT_CKPT)
    ckpt = torch.load(OUT_CKPT, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    test_loss, test_auc, test_acc = validate(model, test_loader, bce, device)
    print(f"Test Loss {test_loss:.4f} AUC {test_auc:.4f} Acc {test_acc:.4f}")

if __name__ == "__main__":
    main()


Overwriting train_kaggle.py


In [4]:
!pip install -q timm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 93.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 73.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 5.2 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 14.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 82.8 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installe

In [5]:
import warnings
warnings.filterwarnings("ignore")


In [13]:
!python train_kaggle.py

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 